In [ ]:
from pathlib import Path

import pandas as pd

## Settings

In [ ]:
TIME_TAGS = [
    "20251109144312",
    "20251109144344",
    "20251109144638",
    "20251109144719",
    "20251109144852",
    "20251109144939",
]

## Merge

In [ ]:
time_tags_list = {}
for time_tag in TIME_TAGS:
    dir_paths = list(Path("./batch_output").glob(f"*{time_tag}*"))
    dir_names = []
    for path in dir_paths:
        name = path.name.split(time_tag, 1)[-1]
        if name != "":
            dir_names.append(time_tag + name)
    time_tags_list[time_tag] = dir_names

print(time_tags_list)

In [ ]:
for time_tag, dir_names in time_tags_list.items():
    print(f"time tag: {time_tag}")

    # Make directories for merged output
    results_final_dir = Path("./results_final") / f"results_{time_tag}"
    results_final_dir.mkdir(parents=True, exist_ok=True)

    # Merge batch_output_logs and results CSVs
    # Read batch_output_logs
    file_path = Path("./batch_output") / f"batch_output_{time_tag}" / "batch_output_logs.csv"
    batch_output_logs = pd.read_csv(file_path)
    batch_output_logs = batch_output_logs.drop(columns=["n_samples_output", "batch_id"])

    # Read result CSVs
    result_dfs = {}
    for _, batch_output_row in batch_output_logs.iterrows():
        dataset_name = batch_output_row["dataset_name"]
        model = batch_output_row["model"]
        repeat = batch_output_row["repeat"]
        forward_or_reverse = batch_output_row["forward_or_reverse"]
        file_path = (
            Path("./results")
            / f"results_{time_tag}"
            / f"dataset_{dataset_name}_{model}_rep{repeat}_{forward_or_reverse}_pair_result.csv"
        )
        result_df_temp = pd.read_csv(file_path)
        result_dfs[(dataset_name, model, repeat, forward_or_reverse)] = result_df_temp

    if dir_names:
        for dir_name in dir_names:
            # Read batch_output_logs from other directories
            file_path = Path("./batch_output") / f"batch_output_{dir_name}" / "batch_output_logs.csv"
            batch_output_logs_temp = pd.read_csv(file_path)
            batch_output_logs = pd.merge(
                batch_output_logs,
                batch_output_logs_temp,
                on=["dataset_name", "model", "repeat", "forward_or_reverse"],
                how="left",
                suffixes=("", "_dup"),
            )

            # input_tokens_mean
            mask = ~batch_output_logs["n_samples_output_succeed_dup"].isna()
            n1 = batch_output_logs.loc[mask, "n_samples_output_succeed"]
            n2 = batch_output_logs.loc[mask, "n_samples_output_succeed_dup"]
            v1 = batch_output_logs.loc[mask, "input_tokens_mean"]
            v2 = batch_output_logs.loc[mask, "input_tokens_mean_dup"]
            batch_output_logs.loc[mask, "input_tokens_mean"] = (v1 * n1 + v2 * n2) / (n1 + n2)

            # input_tokens_min
            batch_output_logs.loc[mask, "input_tokens_min"] = batch_output_logs.loc[
                mask, ["input_tokens_min", "input_tokens_min_dup"]
            ].min(axis=1)

            # input_tokens_max
            batch_output_logs.loc[mask, "input_tokens_max"] = batch_output_logs.loc[
                mask, ["input_tokens_max", "input_tokens_max_dup"]
            ].max(axis=1)

            # input_tokens_sum
            batch_output_logs.loc[mask, "input_tokens_sum"] = batch_output_logs.loc[
                mask, ["input_tokens_sum", "input_tokens_sum_dup"]
            ].sum(axis=1)

            # output_tokens_mean
            v1 = batch_output_logs.loc[mask, "output_tokens_mean"]
            v2 = batch_output_logs.loc[mask, "output_tokens_mean_dup"]
            batch_output_logs.loc[mask, "output_tokens_mean"] = (v1 * n1 + v2 * n2) / (n1 + n2)

            # output_tokens_min
            batch_output_logs.loc[mask, "output_tokens_min"] = batch_output_logs.loc[
                mask, ["output_tokens_min", "output_tokens_min_dup"]
            ].min(axis=1)

            # output_tokens_max
            batch_output_logs.loc[mask, "output_tokens_max"] = batch_output_logs.loc[
                mask, ["output_tokens_max", "output_tokens_max_dup"]
            ].max(axis=1)

            # output_tokens_sum
            batch_output_logs.loc[mask, "output_tokens_sum"] = batch_output_logs.loc[
                mask, ["output_tokens_sum", "output_tokens_sum_dup"]
            ].sum(axis=1)

            # cost_mean
            v1 = batch_output_logs.loc[mask, "cost_mean"]
            v2 = batch_output_logs.loc[mask, "cost_mean_dup"]
            batch_output_logs.loc[mask, "cost_mean"] = (v1 * n1 + v2 * n2) / (n1 + n2)

            # cost_sum
            batch_output_logs.loc[mask, "cost_sum"] = batch_output_logs.loc[mask, ["cost_sum", "cost_sum_dup"]].sum(
                axis=1
            )

            # n_samples_output_succeed
            batch_output_logs.loc[mask, "n_samples_output_succeed"] = n1 + n2

            # Drop duplicate columns
            batch_output_logs = batch_output_logs.drop(
                columns=[col for col in batch_output_logs.columns if col.endswith("_dup")]
            )

            # Read result CSVs from other directories
            for _, batch_output_row in batch_output_logs_temp.iterrows():
                dataset_name = batch_output_row["dataset_name"]
                model = batch_output_row["model"]
                repeat = batch_output_row["repeat"]
                forward_or_reverse = batch_output_row["forward_or_reverse"]
                file_path = (
                    Path("./results")
                    / f"results_{dir_name}"
                    / f"dataset_{dataset_name}_{model}_rep{repeat}_{forward_or_reverse}_pair_result.csv"
                )
                result_df_temp = pd.read_csv(file_path)
                result_dfs[(dataset_name, model, repeat, forward_or_reverse)] = pd.concat(
                    [result_dfs[(dataset_name, model, repeat, forward_or_reverse)], result_df_temp], ignore_index=True
                )

    # Verify merged results
    for _, batch_output_row in batch_output_logs.iterrows():
        dataset_name = batch_output_row["dataset_name"]
        model = batch_output_row["model"]
        repeat = batch_output_row["repeat"]
        forward_or_reverse = batch_output_row["forward_or_reverse"]
        n_samples_input = batch_output_row["n_samples_input"]
        n_samples_output_succeed = batch_output_row["n_samples_output_succeed"]
        result_df = result_dfs[(dataset_name, model, repeat, forward_or_reverse)]
        if not (len(result_df) == n_samples_output_succeed) and (n_samples_input == n_samples_output_succeed):
            raise ValueError(
                f"Number of samples in result_df does not match n_samples_output_succeed for dataset {dataset_name} and model {model} after merging."
            )

    # Save merged CSV
    batch_output_logs.to_csv(results_final_dir / "batch_output_logs.csv", index=False)
    for (dataset_name, model, repeat, forward_or_reverse), result_df in result_dfs.items():
        result_df.to_csv(
            results_final_dir / f"dataset_{dataset_name}_{model}_rep{repeat}_{forward_or_reverse}_pair_result.csv",
            index=False,
        )